# BETO frente a TF-IDF — referencias v2, cinco grupos fijos

**Preparado, no ejecutado con BETO todavía.** Este notebook debe superar la prueba técnica real antes del entrenamiento. No hay una mejora demostrada. No cambies referencias, parámetros ni longitud para que termine.

1. Elige **Entorno de ejecución → Cambiar tipo → GPU** (T4 si está disponible).
2. Ejecuta en orden. La primera instalación es grande; se usa un entorno aislado.
3. Conserva los ZIP descargados y devuelve el último al agente para analizarlo.

No contrates un plan: si no hay GPU gratuita disponible, detente. No se monta Drive ni se solicitan claves HF/GitHub. Colab puede cortar la sesión. No hace falta anotar más textos.

[Guía y recuperación](https://github.com/joako0o/FASE_2/blob/arena/01a0a81b-fase-2/docs/GUIA_EJECUTAR_BETO_COLAB_V1.md)

## 1. Código fijado e instalación aislada

Descarga una revisión pública exacta del proyecto. No se clona ni cambia ninguna rama. Si esta celda falla, no continúes; devuelve el error sin credenciales.

In [ ]:
import io, json, os, subprocess, sys, urllib.request, zipfile
from pathlib import Path

REVISION = "8a088fc3fc283c9e2cccaf179ea3261f61bccd1d"
BASE = Path("/content/fase2_beto_v1")
try:
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)
except (FileNotFoundError, subprocess.CalledProcessError) as exc:
    raise RuntimeError("Activa una GPU gratuita antes de instalar. No contratar un plan.") from exc

BASE.mkdir(parents=True, exist_ok=True)
marca = BASE / "codigo.json"
if marca.exists():
    guardado = json.loads(marca.read_text())
    if guardado["revision"] != REVISION:
        raise RuntimeError("Código de otra revisión: no mezclar experimentos.")
    REPO = BASE / "codigo" / guardado["carpeta"]
    if not (REPO / "scripts/38_preparar_beto.py").is_file():
        raise RuntimeError("Código incompleto; conservar la sesión y consultar el error.")
else:
    destino = BASE / "codigo"
    if destino.exists():
        raise RuntimeError("Descarga parcial existente: no sobrescribir automáticamente.")
    url = f"https://api.github.com/repos/joako0o/FASE_2/zipball/{REVISION}"
    req = urllib.request.Request(url, headers={"User-Agent": "FASE2-BETO-v1"})
    with urllib.request.urlopen(req, timeout=120) as response:
        contenido = response.read()
    with zipfile.ZipFile(io.BytesIO(contenido)) as z:
        nombres = z.namelist()
        raices = {Path(n).parts[0] for n in nombres if Path(n).parts}
        if len(raices) != 1:
            raise ValueError("Archivo de código sin raíz única.")
        for n in nombres:
            if not (destino / n).resolve().is_relative_to(destino.resolve()):
                raise ValueError("Ruta insegura en archivo ZIP.")
        carpeta = raices.pop()
        z.extractall(destino)
    REPO = destino / carpeta
    marca.write_text(json.dumps({"revision": REVISION, "carpeta": carpeta}))

ENV = BASE / "entorno"
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "virtualenv==20.35.3"], check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(ENV)], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "install", "-r", str(REPO / "requirements-beto.txt")], check=True)
subprocess.run([str(PYTHON), "-m", "pip", "check"], check=True)
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"

def ejecutar(script, *args):
    subprocess.run([str(PYTHON), "-u", str(REPO / "scripts" / script), *map(str, args)], cwd=REPO, check=True)

ENTRADA = REPO / "data/checkpoints/beto_v1/entrada"
RESULTADOS = BASE / "ejecucion"
CACHE = BASE / "pesos"
print("Código fijado:", REVISION)

## 2. Preparación, controles y disponibilidad

Verifica los 1.352 textos, cinco particiones y control TF-IDF. Los tests de esta celda **no ejecutan el encoder**: incluyen un tokenizador de juguete y resultados sintéticos temporales para probar el comparador. No son métricas BETO.

In [ ]:
if ENTRADA.exists():
    ejecutar("38_preparar_beto.py", "--verificar")
else:
    ejecutar("38_preparar_beto.py")
subprocess.run([str(PYTHON), "-m", "unittest", "discover", "-s", "tests", "-p", "test_preparacion_beto.py", "-v"], cwd=REPO, check=True)
ejecutar("39_ejecutar_beto.py", "--comprobar")
PAQUETE_ID = json.loads((ENTRADA / "manifest.json").read_text())["paquete_id"]
print("Paquete:", PAQUETE_ID)

### Opcional: recuperar un respaldo anterior

**Primera ejecución: dejar `False`.** Solo en una sesión nueva, después de las dos celdas anteriores, cambiar a `True` para subir un único `beto_resultados.zip`. No sobrescribe una carpeta de resultados existente. La prueba técnica y los grupos completos se comprobarán después. No recupera entrenamiento a mitad de un grupo.

In [ ]:
RESTAURAR_ZIP = False
if RESTAURAR_ZIP:
    from google.colab import files
    if RESULTADOS.exists():
        raise FileExistsError("Ya hay resultados: no mezclar ni sobrescribir.")
    subidos = files.upload()
    if len(subidos) != 1:
        raise ValueError("Sube exactamente un ZIP, el respaldo más reciente.")
    with zipfile.ZipFile(io.BytesIO(next(iter(subidos.values())))) as z:
        if sum(i.file_size for i in z.infolist()) > 25_000_000:
            raise ValueError("Respaldo demasiado grande: no se esperan pesos ni corpus.")
        for i in z.infolist():
            p = Path(i.filename)
            if not p.parts or p.parts[0] != "ejecucion" or not (BASE / p).resolve().is_relative_to(RESULTADOS.resolve()):
                raise ValueError("Ruta fuera de resultados.")
            if not i.is_dir() and p.suffix not in {".json", ".jsonl", ".csv"}:
                raise ValueError("Archivo no esperado en respaldo.")
        z.extractall(BASE)
    print("Respaldo recuperado; aún debe pasar las verificaciones.")

## 3. Prueba técnica con pesos oficiales

Descarga el checkpoint fijado y verifica sus hashes. Ejecuta un paso de entrenamiento con dos intervenciones completas de train, incluida una larga; verifica pérdida/gradientes, cambios de cabeza y encoder. La instancia se descarta antes de los cinco grupos. **Si falla, detenerse y enviar el error.** No omitir ni falsificar `smoke.json`.

In [ ]:
smoke = RESULTADOS / "smoke.json"
if smoke.exists():
    previo = json.loads(smoke.read_text())
    if previo.get("estado") != "aprobado_con_pesos_reales" or previo.get("paquete_id") != PAQUETE_ID:
        raise RuntimeError("Prueba técnica incompatible; no continuar.")
    print("Registro de prueba técnica previa del mismo paquete:")
    print(json.dumps(previo, ensure_ascii=False, indent=2))
else:
    ejecutar("39_ejecutar_beto.py", "--smoke", "--salida", RESULTADOS, "--cache", CACHE)

## 4. Entrenar y evaluar los cinco grupos

Tres épocas fijas por grupo, sin seleccionar la época con validación. Se preserva el filtro A del control. Los grupos completos con manifiestos válidos no se repiten. Una carpeta parcial causa una parada, no una sobrescritura.

Se descarga un ZIP al terminar cada grupo: permite descargas múltiples y conserva el más reciente. No basta con ver un archivo en Colab: la sesión puede perderlo. No se guardan pesos finales; este experimento compara modelos, no reemplaza el de producción.

In [ ]:
from google.colab import files

def respaldar():
    archivo = BASE / "beto_resultados.zip"
    with zipfile.ZipFile(archivo, "w", compression=zipfile.ZIP_DEFLATED) as z:
        for p in sorted(RESULTADOS.rglob("*")):
            if p.is_file():
                z.write(p, arcname=p.relative_to(BASE))
    print("Respaldo:", archivo, "— conservar la descarga fuera de Colab.")
    files.download(str(archivo))

for fold in range(1, 6):
    print(f"=== Grupo {fold}/5 ===", flush=True)
    try:
        ejecutar("39_ejecutar_beto.py", "--fold", fold, "--reanudar", "--salida", RESULTADOS, "--cache", CACHE)
    except subprocess.CalledProcessError:
        if RESULTADOS.exists():
            respaldar()
        raise
    respaldar()

## 5. Comparación completa y entrega

Solo se genera después de verificar los cinco grupos. Compara F1 H/D, recalls, H→D, D→H, H/D→N, N→H/D y matriz completa. Desglosa las cinco inversiones ambiguas conocidas sin quitarlas del resultado principal. **Desarrollo reutilizado, no test independiente ni adopción automática.**

Descarga el ZIP final y devuélvelo al agente. No interpretar las pérdidas de entrenamiento como precisión.

In [ ]:
comparacion = RESULTADOS / "comparacion.json"
if comparacion.exists():
    print("Comparación previa conservada; no se vuelve a escribir.")
else:
    ejecutar("39_ejecutar_beto.py", "--consolidar", "--salida", RESULTADOS)
resumen = json.loads(comparacion.read_text())
if resumen.get("paquete_id") != PAQUETE_ID:
    raise RuntimeError("Comparación de otro paquete; no mezclar resultados.")
for nombre, r in resumen["condiciones"].items():
    conjunto = r["conjunto"]
    print(nombre, {"F1_HD_medio": r["media_f1_hd"], "H_a_D": conjunto["h_a_d"],
                   "D_a_H": conjunto["d_a_h"], "HD_a_N": conjunto["hd_a_n"], "N_a_HD": conjunto["n_a_hd"]})
print("Criterios:", resumen["criterios"])
print("¿Cumple criterios de desarrollo?", resumen["cumple_criterios_desarrollo"])
print("No prueba generalización ni adopta el modelo automáticamente.")
respaldar()